# Part 16: Deep Learning

**Quick Reference for Neural Networks & Deep Learning**

[Back to Index](Index.ipynb)

---
## 16.1 Artificial Neural Networks (ANN)

**Architecture:** Input layer → Hidden layers → Output layer

**Forward Propagation:** z = Wx + b, a = activation(z)

**Backpropagation:** Compute gradients and update weights

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# Load data
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for neural networks!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build ANN
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dropout(0.3),  # Regularization
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

# Train
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.3f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training Accuracy')
plt.tight_layout()
plt.show()

### Activation Functions

In [ ]:
# Common activation functions:
# 1. ReLU (Rectified Linear Unit): max(0, x) - Most common for hidden layers
# 2. Sigmoid: 1/(1+e^-x) - Binary classification output
# 3. Tanh: (e^x - e^-x)/(e^x + e^-x) - Alternative to sigmoid
# 4. Softmax: e^xi / Σe^xj - Multi-class classification output
# 5. Leaky ReLU: max(0.01x, x) - Solves dying ReLU problem

# Visualize
x = np.linspace(-5, 5, 100)
relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
leaky_relu = np.where(x > 0, x, 0.01 * x)

plt.figure(figsize=(12, 4))
plt.plot(x, relu, label='ReLU')
plt.plot(x, sigmoid, label='Sigmoid')
plt.plot(x, tanh, label='Tanh')
plt.plot(x, leaky_relu, label='Leaky ReLU')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.title('Activation Functions')
plt.show()

---
## 16.2 Convolutional Neural Networks (CNN)

**Purpose:** Process grid-like data (images)

**Key Layers:**
- Conv2D: Extract features
- MaxPooling: Reduce dimensions
- Flatten: Convert to 1D
- Dense: Classification

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocess
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Build CNN
model = keras.Sequential([
    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Conv Block 3
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Classifier
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

# Train
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.3f}")

# Visualize predictions
predictions = model.predict(X_test[:9])
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    pred_label = np.argmax(predictions[i])
    true_label = np.argmax(y_test[i])
    plt.title(f"Pred: {pred_label}, True: {true_label}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### Transfer Learning with Pre-trained Models

In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load pre-trained model (without top layers)
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

# Add custom classifier
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

# Train
# model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=10)

---
## 16.3 Recurrent Neural Networks (RNN)

**Purpose:** Process sequential data (text, time series)

**Concept:** Hidden state maintains information from previous time steps

**Problem:** Vanishing gradient (solved by LSTM/GRU)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, SimpleRNN

# Example: Text classification
max_features = 10000  # vocabulary size
maxlen = 100  # sequence length

# Build RNN
model = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.SimpleRNN(64, return_sequences=False),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

---
## 16.4 Long Short-Term Memory (LSTM)

**Purpose:** Solve vanishing gradient problem in RNNs

**Components:**
- Forget gate: What to forget
- Input gate: What to store
- Output gate: What to output
- Cell state: Long-term memory

In [ ]:
from tensorflow.keras.layers import LSTM, Bidirectional, GRU

# LSTM for text classification
model = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.LSTM(64, return_sequences=True),  # Stacked LSTM
    layers.LSTM(32),
    layers.Dense(1, activation='sigmoid')
])

# Bidirectional LSTM (process forward and backward)
model_bi = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(1, activation='sigmoid')
])

# GRU (simpler alternative to LSTM, faster)
model_gru = keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.GRU(64),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

### LSTM for Time Series Forecasting

In [ ]:
# Prepare time series data
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Example: Predict next value
seq_length = 10
X, y = create_sequences(time_series_data, seq_length)
X = X.reshape(X.shape[0], X.shape[1], 1)  # Add feature dimension

# Build LSTM model
model = keras.Sequential([
    layers.LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    layers.LSTM(50, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

# Predict
predictions = model.predict(X_test)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(y_test, label='Actual')
plt.plot(predictions, label='Predicted')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.title('LSTM Time Series Forecasting')
plt.show()

---
## 16.5 Advanced Architectures

### Transformers (Attention Mechanism)

In [ ]:
# Self-attention mechanism
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        self.dense = layers.Dense(d_model)
    
    def call(self, query, key, value):
        # Simplified attention implementation
        q = self.wq(query)
        k = self.wk(key)
        v = self.wv(value)
        
        attention_weights = tf.nn.softmax(tf.matmul(q, k, transpose_b=True))
        output = tf.matmul(attention_weights, v)
        
        return self.dense(output)

# Transformer block
def transformer_block(inputs, d_model, num_heads, ff_dim, dropout=0.1):
    # Multi-head attention
    attn_output = MultiHeadAttention(d_model, num_heads)(inputs, inputs, inputs)
    attn_output = layers.Dropout(dropout)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)
    
    # Feed forward
    ffn_output = layers.Dense(ff_dim, activation='relu')(out1)
    ffn_output = layers.Dense(d_model)(ffn_output)
    ffn_output = layers.Dropout(dropout)(ffn_output)
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)
    
    return out2

### Autoencoders

In [ ]:
# Autoencoder for dimensionality reduction
input_dim = 784  # e.g., MNIST flattened
encoding_dim = 32

# Encoder
encoder_input = layers.Input(shape=(input_dim,))
encoded = layers.Dense(128, activation='relu')(encoder_input)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

# Decoder
decoded = layers.Dense(128, activation='relu')(encoded)
decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)

# Autoencoder model
autoencoder = keras.Model(encoder_input, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Encoder model (for getting encodings)
encoder = keras.Model(encoder_input, encoded)

# Train
# autoencoder.fit(X_train, X_train, epochs=50, batch_size=256, validation_split=0.2)

# Get encodings
# encoded_data = encoder.predict(X_test)

### Variational Autoencoders (VAE)

In [ ]:
# VAE - generates new samples
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
latent_dim = 2
encoder_inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')

# Decoder
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation='relu')(latent_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name='decoder')

### Generative Adversarial Networks (GANs)

In [ ]:
# Simple GAN structure
latent_dim = 100

# Generator
generator = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(latent_dim,)),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(784, activation='tanh'),
    layers.Reshape((28, 28, 1))
])

# Discriminator
discriminator = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# GAN model
discriminator.trainable = False
gan_input = layers.Input(shape=(latent_dim,))
generated_image = generator(gan_input)
gan_output = discriminator(generated_image)
gan = keras.Model(gan_input, gan_output)
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Training loop (simplified)
# for epoch in range(epochs):
#     # Train discriminator
#     noise = np.random.normal(0, 1, (batch_size, latent_dim))
#     fake_images = generator.predict(noise)
#     d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))
#     d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size, 1)))
#     
#     # Train generator
#     noise = np.random.normal(0, 1, (batch_size, latent_dim))
#     g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

---
## 16.6 PyTorch Alternative

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Initialize
model = NeuralNetwork()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(num_epochs):
    for batch_idx, (data, target) in enumerate(train_loader):
        # Forward
        output = model(data)
        loss = criterion(output, target)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')

---
### Quick Reference

**Architecture Comparison:**

| Architecture | Use Case | Input | Key Feature |
|--------------|----------|-------|-------------|
| ANN | Tabular data | Vectors | Fully connected |
| CNN | Images | Grids | Convolutional layers |
| RNN | Sequences | Time series | Recurrence |
| LSTM | Long sequences | Text, time series | Gates |
| Transformer | NLP | Text | Attention |
| Autoencoder | Compression | Any | Encoding-Decoding |
| GAN | Generation | Noise | Adversarial training |

**Common Hyperparameters:**
- **Learning rate:** 0.001 (Adam), 0.01 (SGD)
- **Batch size:** 32, 64, 128
- **Epochs:** Early stopping with patience
- **Dropout:** 0.2-0.5
- **Optimizer:** Adam (most common), SGD, RMSprop

**Regularization Techniques:**
- Dropout
- L1/L2 regularization
- Batch normalization
- Data augmentation
- Early stopping

**TensorFlow vs PyTorch:**
- TensorFlow: Production, deployment, Keras API
- PyTorch: Research, flexibility, dynamic graphs

**Best Practices:**
- Always scale/normalize inputs
- Use GPU for training (CUDA)
- Monitor train/val loss for overfitting
- Use callbacks (early stopping, model checkpoint)
- Start simple, then increase complexity
- Use pre-trained models when possible